# Clustering Molecules with Machine Learning

Can a neural network learn to group molecules by chemical family **without being told the answer**? In this notebook, we'll:

1. Compute **molecular descriptors** for 30+ compounds across 5 families
2. Train a **PyTorch autoencoder** to compress those features to 2D
3. Visualize the results as a scatter plot where **similar molecules cluster together**

This is the **same architecture** used in the companion Birdsong project (clustering bird species by song) and the Climate project (clustering cities by temperature patterns).

## Step 1: Import Libraries

In [ ]:
!pip install rdkit-pypi -q

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, DataStructs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

## Step 2: Build the Compound Library

In [ ]:
compound_library = [
    # Alcohols
    {"name": "Methanol",       "smiles": "CO",                     "family": "Alcohol"},
    {"name": "Ethanol",        "smiles": "CCO",                    "family": "Alcohol"},
    {"name": "1-Propanol",     "smiles": "CCCO",                   "family": "Alcohol"},
    {"name": "1-Butanol",      "smiles": "CCCCO",                  "family": "Alcohol"},
    {"name": "1-Pentanol",     "smiles": "CCCCCO",                 "family": "Alcohol"},
    {"name": "Isopropanol",    "smiles": "CC(C)O",                 "family": "Alcohol"},
    {"name": "Cyclohexanol",   "smiles": "OC1CCCCC1",              "family": "Alcohol"},
    # Carboxylic Acids
    {"name": "Formic Acid",    "smiles": "O=CO",                   "family": "Acid"},
    {"name": "Acetic Acid",    "smiles": "CC(=O)O",                "family": "Acid"},
    {"name": "Propionic Acid", "smiles": "CCC(=O)O",               "family": "Acid"},
    {"name": "Butyric Acid",   "smiles": "CCCC(=O)O",              "family": "Acid"},
    {"name": "Pentanoic Acid", "smiles": "CCCCC(=O)O",             "family": "Acid"},
    {"name": "Benzoic Acid",   "smiles": "OC(=O)c1ccccc1",         "family": "Acid"},
    # Aromatics
    {"name": "Benzene",        "smiles": "c1ccccc1",               "family": "Aromatic"},
    {"name": "Toluene",        "smiles": "Cc1ccccc1",              "family": "Aromatic"},
    {"name": "Naphthalene",    "smiles": "c1ccc2ccccc2c1",         "family": "Aromatic"},
    {"name": "Aniline",        "smiles": "Nc1ccccc1",              "family": "Aromatic"},
    {"name": "Phenol",         "smiles": "Oc1ccccc1",              "family": "Aromatic"},
    {"name": "Styrene",        "smiles": "C=Cc1ccccc1",            "family": "Aromatic"},
    {"name": "Anthracene",     "smiles": "c1ccc2cc3ccccc3cc2c1",   "family": "Aromatic"},
    # Amines
    {"name": "Methylamine",    "smiles": "CN",                     "family": "Amine"},
    {"name": "Ethylamine",     "smiles": "CCN",                    "family": "Amine"},
    {"name": "Dimethylamine",  "smiles": "CNC",                    "family": "Amine"},
    {"name": "Trimethylamine", "smiles": "CN(C)C",                 "family": "Amine"},
    {"name": "Propylamine",    "smiles": "CCCN",                   "family": "Amine"},
    {"name": "Diethylamine",   "smiles": "CCNCC",                  "family": "Amine"},
    # Drug-like
    {"name": "Aspirin",        "smiles": "CC(=O)Oc1ccccc1C(=O)O",  "family": "Drug"},
    {"name": "Ibuprofen",      "smiles": "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "family": "Drug"},
    {"name": "Acetaminophen",  "smiles": "CC(=O)Nc1ccc(O)cc1",     "family": "Drug"},
    {"name": "Caffeine",       "smiles": "Cn1c(=O)c2c(ncn2C)n(C)c1=O", "family": "Drug"},
    {"name": "Naproxen",       "smiles": "COc1ccc2cc(ccc2c1)C(C)C(=O)O", "family": "Drug"},
    {"name": "Lidocaine",      "smiles": "CCN(CC)CC(=O)Nc1c(C)cccc1C", "family": "Drug"},
]

print(f"Library: {len(compound_library)} compounds")

## Step 3: Feature Extraction — Molecular "Fingerprint Vector"

We'll compute **26 molecular descriptors** per compound — matching the 26 features used in the Birdsong (MFCCs) and Climate (monthly means) projects.

| Features | Count |
|----------|-------|
| Morgan fingerprint PCA components | 10 |
| MW, LogP, TPSA, HBD, HBA | 5 |
| RotBonds, AromaticRings, FractionCSP3 | 3 |
| NumAtoms, NumBonds, RingCount | 3 |
| BertzCT (complexity), LabuteASA, PEOE_VSA | 3 |
| Chi0v, Kappa1 | 2 |
| **Total** | **26** |

In [ ]:
from sklearn.decomposition import PCA

def compute_features(smiles):
    """Compute 26 molecular features from a SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Morgan fingerprint -> PCA will reduce to 10 components later
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=512)
    fp_arr = np.zeros(512)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)

    # Physicochemical descriptors (16 features)
    desc = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.FractionCSP3(mol),
        mol.GetNumHeavyAtoms(),
        mol.GetNumBonds(),
        Descriptors.RingCount(mol),
        Descriptors.BertzCT(mol),
        Descriptors.LabuteASA(mol),
        Descriptors.PEOE_VSA1(mol),
        Descriptors.Chi0v(mol),
        Descriptors.Kappa1(mol),
    ]

    return fp_arr, np.array(desc)

# Compute features
fp_list = []
desc_list = []
names = []
families = []

for c in compound_library:
    result = compute_features(c["smiles"])
    if result:
        fp_list.append(result[0])
        desc_list.append(result[1])
        names.append(c["name"])
        families.append(c["family"])

# PCA on fingerprints to get 10 components
fp_matrix = np.array(fp_list)
pca = PCA(n_components=10)
fp_pca = pca.fit_transform(fp_matrix)

# Combine: 10 FP components + 16 descriptors = 26 features
desc_matrix = np.array(desc_list)
feature_matrix = np.hstack([fp_pca, desc_matrix])

print(f"Feature matrix: {feature_matrix.shape[0]} compounds x {feature_matrix.shape[1]} features")

## Step 4: Preprocessing

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_matrix)
X_tensor = torch.FloatTensor(X_scaled)

print(f"Tensor shape: {X_tensor.shape}")

## Step 5: The Autoencoder

Same architecture as the Birdsong and Climate projects:

```
Input (26 features) -> 128 -> 64 -> 2 (bottleneck) -> 64 -> 128 -> Output (26 features)
```

The bottleneck forces the network to find the 2 most important dimensions for distinguishing molecules.

In [ ]:
class MoleculeAutoencoder(nn.Module):
    def __init__(self, input_dim=26, bottleneck=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = MoleculeAutoencoder(input_dim=26, bottleneck=2)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 6: Train

In [ ]:
LEARNING_RATE = 0.0001
EPOCHS = 5000

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

losses = []
for epoch in range(EPOCHS):
    output = model(X_tensor)
    loss = criterion(output, X_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        losses.append(loss.item())
    if epoch % 1000 == 0:
        print(f"  Epoch {epoch:5d}: loss = {loss.item():.6f}")

print(f"  Final loss: {loss.item():.6f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(0, EPOCHS, 100), losses, color="steelblue")
plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True, alpha=0.3)
plt.show()

## Step 7: Visualize the Clusters

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model.encoder(X_tensor).numpy()

family_colors = {
    "Alcohol": "#3498db", "Acid": "#e74c3c", "Aromatic": "#9b59b6",
    "Amine": "#2ecc71", "Drug": "#f39c12"
}

plt.figure(figsize=(12, 9))

for i, name in enumerate(names):
    color = family_colors[families[i]]
    plt.scatter(embeddings[i, 0], embeddings[i, 1], c=color, s=100,
                zorder=3, edgecolors="black", linewidth=0.5)
    plt.annotate(name, (embeddings[i, 0], embeddings[i, 1]),
                 textcoords="offset points", xytext=(6, 6), fontsize=7)

for fam, color in family_colors.items():
    plt.scatter([], [], c=color, s=80, label=fam, edgecolors="black", linewidth=0.5)
plt.legend(title="Chemical Family", loc="best")

plt.title("Molecular Clustering (Autoencoder 2D Embedding)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.show()

## Step 8: Interpret the Results

Look at the scatter plot and consider:
- **Do alcohols cluster together?** Acids? Amines?
- **Where do the drugs land?** They have aromatic rings AND polar groups — do they overlap with aromatics and acids?
- **Which families are most similar?** Which are most distinct?
- **Does chain length matter?** Methanol vs. pentanol — close or far apart?

The autoencoder discovered these groupings without being told the family labels!

## Step 9: Experiment!

Try changing these hyperparameters:

| Parameter | Default | Try |
|-----------|---------|-----|
| `LEARNING_RATE` | 0.0001 | 0.001, 0.00001 |
| `EPOCHS` | 5000 | 500, 10000 |
| `bottleneck` | 2 | 3, 5 |

In [ ]:
# === EXPERIMENT ===
LEARNING_RATE = 0.0001
EPOCHS = 5000
BOTTLENECK = 2

model_exp = MoleculeAutoencoder(input_dim=26, bottleneck=BOTTLENECK)
opt_exp = torch.optim.Adam(model_exp.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    out = model_exp(X_tensor)
    loss = nn.MSELoss()(out, X_tensor)
    opt_exp.zero_grad()
    loss.backward()
    opt_exp.step()

print(f"Final loss: {loss.item():.6f}")

model_exp.eval()
with torch.no_grad():
    emb = model_exp.encoder(X_tensor).numpy()

plt.figure(figsize=(12, 9))
for i, name in enumerate(names):
    color = family_colors[families[i]]
    plt.scatter(emb[i, 0], emb[i, 1], c=color, s=100, zorder=3,
                edgecolors="black", linewidth=0.5)
    plt.annotate(name, (emb[i, 0], emb[i, 1]),
                 textcoords="offset points", xytext=(6, 6), fontsize=7)
for fam, color in family_colors.items():
    plt.scatter([], [], c=color, s=80, label=fam, edgecolors="black", linewidth=0.5)
plt.legend(title="Chemical Family", loc="best")
plt.title(f"Experiment: lr={LEARNING_RATE}, epochs={EPOCHS}, bottleneck={BOTTLENECK}")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.show()

## What's Next

In **Notebook 4**, we'll explore **3D molecular structures** and **chemical dynamics** — visualizing molecules in 3D and simulating reaction kinetics with differential equations.